# Block Stacking Problem

In [ ]:
#@title Verification code

import numpy as np
import sys

FLOAT_TOLERANCE = sys.float_info.epsilon * 100


def get_positions_score(positions: list[float]) -> float:
  """Calculates the score of given positions (block right endpoints).

  Each block has width 1. The position value represents the right endpoint
  of each block. The blocks are stacked from bottom to top, and the
  constraint is that the center of mass of all blocks above block i must
  lie within the support of block i (between positions[i]-1 and positions[i]).

  The score is the rightmost endpoint (positions[-1]), representing
  the maximum overhang achieved.
  """
  n = len(positions)
  if n == 0:
    return 0.0

  if n == 1:
    if positions[0] - 0.5 >= 0.0 - FLOAT_TOLERANCE:
      return -1.0
    return positions[0]

  sum_all = 0.0
  for k in range(n):
    sum_all += (positions[k] - 0.5)
  sum_all_avg = sum_all / n
  if sum_all_avg >= 0.0 - FLOAT_TOLERANCE:
    return -1.0

  upper_sum = 0.0
  upper_count = 0.0

  if n > 1:
    upper_sum = positions[n-1] - 0.5
    upper_count = 1.0

  for i in range(n - 2, -1, -1):
    upper_sum_avg = upper_sum / upper_count
    lb = positions[i] - 1.0
    ub = positions[i]
    if not (lb - FLOAT_TOLERANCE <= upper_sum_avg <= ub + FLOAT_TOLERANCE):
      return -1.0
    upper_sum += (positions[i] - 0.5)
    upper_count += 1.0
  return positions[-1]


FUNCTION_IS_TOO_LARGE = 101.0
WRONG_OUTPUT_SHAPE = 102.0


def get_score_of_algorithm(solve_fn):
  """Returns the score of the given algorithm over a range of n values."""
  all_scores = []
  for n in range(2, 50):
    curr_positions = solve_fn(n)
    if len(curr_positions) != n:
      all_scores.append(-WRONG_OUTPUT_SHAPE)
      continue
    if np.max(np.abs(curr_positions)) > 1e3:
      all_scores.append(-FUNCTION_IS_TOO_LARGE)
      continue
    curr_score = get_positions_score(curr_positions)
    all_scores.append(curr_score)
  return all_scores

In [ ]:
#@title Initial program

import numpy as np


def get_positions(n: int) -> list[float]:
  """Returns n block positions maximizing overhang.

  Each position represents the right endpoint of a unit-width block.
  Blocks are stacked bottom to top, and the center of mass of all blocks
  above block i must lie within the support [positions[i]-1, positions[i]].

  The classic harmonic stacking gives overhang sum_{k=1}^{n} 1/(2k),
  but better constructions exist using counterweighting.
  """
  # Harmonic stacking baseline: each block extends 1/(2k) beyond the one above
  positions = []
  offset = 0.0
  for k in range(n, 0, -1):
    offset += 1.0 / (2 * k)
    positions.append(offset)
  positions.reverse()
  return positions


# Verify the solution
scores = get_score_of_algorithm(get_positions)
total = sum(s for s in scores if s > 0)
print(f"Total score (sum of valid overhangs for n=2..49): {total:.6f}")
for n_val, score in zip(range(2, 50), scores):
  if score > 0:
    print(f"  n={n_val}: overhang = {score:.6f}")

**Prompt used**

Act as a research mathematician, software developer and optimization specialist
in constructing lists of floats with certain extremal properties.

GOAL:
Your task is to find a general algorithm which, for a given
natural number n, produces a python list of n positive floats ("positions") that
maximizes the following scoring function:

def get_positions_score(positions: list[float]) -> float:
  """Calculates the score of given points."""
  n = len(positions)
  if n == 0:
    return 0.0

if n == 1:
    if positions[0] - 0.5 >= 0.0 - FLOAT_TOLERANCE:
      return -1.0
    return positions[0]

sum_all = 0.0
  for k in range(n):
    sum_all += (positions[k] - 0.5)
  sum_all_avg = sum_all / n
  if sum_all_avg >= 0.0 - FLOAT_TOLERANCE:
    return -1.0

upper_sum = 0.0
  upper_count = 0.0

if n > 1:
    upper_sum = positions[n-1] - 0.5
    upper_count = 1.0

for i in range(n - 2, -1, -1):
    upper_sum_avg = upper_sum / upper_count
    lb = positions[i] - 1.0
    ub = positions[i]
    if not (lb - FLOAT_TOLERANCE <= upper_sum_avg <= ub + FLOAT_TOLERANCE):
      return -1.0
    upper_sum += (positions[i] - 0.5)
    upper_count += 1.0
  return positions[-1]

Specifically, the Python function that contains your algorithm and you have to
provide is called get_positions() and has the following signature:

def get_positions(n: int) -> list[float]:

HINTS:
1. You will be evaluated against a wide range of small and large values of n, so
you MUST try to find a general solution to the problem. I would strongly
encourage you to try to find a general solution. Your program will be evaluated
on some very large values of n -- try to find the pattern that works for all n.

The previous solution provided in this prompt is still not optimal, much better configurations are possible. The patterns you have to discover are not hard, you can definitely improve it, it is not beyond your capabilities. DO NOT go for the same solution as the previous one. Always try to find a better pattern, don't be scared of the difficult sounding problem, once you see the solution you'll realise it wasn't hard at all. Good luck, I believe in you, but you also have to believe in yourself!
IMPORTANT CONSTRAINTS:
0. Your algorithm needs to be fast!!! If get_sequences() takes more than 15
seconds to run, you will get a penalty.
1. The function get_sequences must be deterministic!
Please keep in mind that if the provided get_sequences function is NOT
deterministic (e.g. uses randomness) your algorithm will receive a penalty
during the scoring.
2. Try to keep get_sequences concise, short and human-readable.


In [ ]:
#@title Code evolved by AlphaEvolve



def get_positions(n: int) -> list[float]:
  """
  Constructs a list of n floats that maximizes the score.

  The method is based on an analytical derivation that makes the inequality
  constraints in the scoring function tight. This leads to a recurrence
  relation p_i = p_{i+1} - 0.5 / (n - 1 - i). The value p_{n-1} is chosen
  to be close to its theoretical maximum, which is derived from the sum
  constraint sum(p_k - 0.5) < 0 and the positivity constraint p_k > 0.
  """
  if n == 0:
    return []
  if n == 1:
    # For n=1, score is positions[0] if positions[0] < 0.5. Maximize it.
    return [0.4999999999]

  # Calculate H_{n-1}, the (n-1)-th harmonic number, using numpy for speed.
  # Use float64 for precision with large n.
  h_n_minus_1 = np.sum(1.0 / np.arange(1, n, dtype=np.float64))

  # Theoretical max for p_{n-1} is 0.5 * H_{n-1} + 0.5 / n. We subtract a small epsilon.
  p_n_minus_1 = 0.5 * h_n_minus_1 + 0.5 / n - 1e-12

  # Using a numpy array is faster for allocation.
  positions = np.zeros(n, dtype=np.float64)
  positions[n - 1] = p_n_minus_1

  # Compute the rest of the positions via the recurrence.
  for i in range(n - 2, -1, -1):
    positions[i] = positions[i+1] - 0.5 / (n - 1.0 - i)

  return positions.tolist()

## What AlphaEvolve found

The well-known answer to this problem is $C(n) = \frac{1}{2} H_n$, where $H_n = 1 + \frac{1}{2} + \cdots + \frac{1}{n}$ is the $n$-th harmonic number. AlphaEvolve was given an obfuscated scoring function and, using generalizer mode, rediscovered the correct solution within one or two iterations. It derived the harmonic number formula by reasoning about the recurrence relation arising from the constraint structure, eventually producing positions of the form $\text{positions}[k] = 0.5 \cdot (H_n - H_{n-k-1})$.